In [ ]:
# Importing Libraries
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler


from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVR

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA

from sklearn.inspection import permutation_importance
from sklearn.metrics import (
accuracy_score, precision_score, f1_score, recall,
classification_report, confusion_matrix, roc_auc_score, roc_curve
)


# Data Collection
df = pd.read_csv("data.csv")
df = pd.read_excel("data.xlsx")
df = pd.read_json("data.json")


# Understanding Data
df.head()
df.tail()
df.info()
df.describe()
df.nunique()
df.shape
df.columns
df.dtypes


# Selecting rows and columns
df["col1"]
df[["col1", "col2"]]
df.loc[0] 
df.loc[0:3] 
df.loc[0, "col"]
df[df["col"] > 100]
df.query("col1 > 100 and col2 < 500")


# Data Cleaning
df.isnull()
df.isnull().sum()
df.dropna()
df.fillna(val)
df.ffill()
df.bfill()

df.duplicated()                         
df.duplicated("country")                 
df.duplicated(["country", "gender"])
df.drop_duplicates()                   
df.astype(new_type)
pd.to_datetime()

df["col"].str.lower()
df["col"].str.upper()
df["col"].str.capitalize()
df["col"].str.strip()
df["col"].str.split(" ")
df["col"].str.contains()

df['Age_plus_10'] = df['Age'].apply(lambda x: x + 10)
df['City'] = df['City'].map({'Delhi':'DL', 'Mumbai':'MB'})
df.assign(new_income = df["income"] * 1.1)
df['City'] = df['City'].replace({'Delhi':'DL', 'Mumbai':'MB'})
df_renamed = df.rename(columns={"A": "X", "B": "Y"}) 
df.sort_values("col1") 
df.groupby("country")["income"].mean()
df.sum()
df.mean()
df.count()
df.max()
df.min()
df.std()
df.aggr()

pd.merge(df_customers, df_orders, on="customer_id") 
pd.merge(df_customers, df_orders, on="customer_id", how="left") 
pd.merge(df_customers, df_orders, on="customer_id", how="right") 
pd.merge(df_customers, df_orders, on="customer_id", how="outer") 
df.plot(kind='scatter', x='age', y='income')


# Data Visualization
plt.figure(figsize=(15, 15))
plt.xlabel("X values")
plt.ylabel("Y values")
plt.title("Simple Line Plot")
plt.grid(True) 
plt.legend() 
plt.savefig("line_plot.png")
plt.tight_layout()
plt.show()

plt.bar(x, y)
plt.barh(x, y) 
plt.scatter(x, y)
plt.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors)
plt.hist(data, bins=5, color='skyblue', edgecolor='black')
plt.boxplot(data)
plt.stackplot(x, y1, y2, labels=['Data1', 'Data2'], colors=['blue', 'green'])


sns.scatterplot(data=tips, x="total_bill", y="tip", hue="day", style="sex", size="size")
sns.lineplot(data=data, x="total_bill", y="tip")
sns.barplot(data=tips, x="day", y="total_bill", hue="sex") 
sns.boxplot(data=tips, x="day", y="total_bill", hue="sex") 
sns.histplot(data=tips, x="total_bill", bins=10, kde=False)
sns.heatmap (corr, annot=True, cmap="coolwarm")


# Train Test Split
X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Model Pipeline
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include=["str"]).columns

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(drop="first",handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, num_features),
    ("categorical", categorical_pipeline, cat_features)
])


# Model Training
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)

# Model Evaluation
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]
print(classification_report(y_test, y_prob))

# Feature Importance
log_model = log_pipeline.named_steps["classifier"]
feature_names = log_pipeline.named_steps["preprocessor"].get_feature_names_out()
importance = pd.DataFrame({"Feature": feature_names, "Coefficient": log_model.coef_[0]})
importance["AbsCoefficient"] = importance["Coefficient"].abs()
importance = importance.sort_values(by="AbsCoefficient", ascending=False)
plt.figure(figsize=(10,6))
sns.barplot(data=importance.head(15), x="AbsCoefficient", y="Feature")
plt.title("Top 15 Logistic Regression Features")
plt.ylabel("Absolute Coefficient")
plt.show()

# Hyperparameter Tuning
param_grid = {
    'classifier__C': [0.1, 1, 10],
    'classifier__gamma': ['scale', 'auto', 0.01, 0.1],
    'classifier__kernel': ['rbf', 'linear']
}

grid_search = GridSearchCV(svm_pipeline, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best SVM Parameters: {grid_search.best_params_}")
best_svm_model = grid_search.best_estimator_


best_pred = best_svm_model.predict(X_test)
best_prob = best_svm_model.predict_proba(X_test)[:, 1]
best_roc = roc_auc_score(y_test, best_prob)
print("Tuned SVM ROC-AUC:", best_roc)


# Final Model Selection
models = {"Logistic Regression": log_pipeline, "Random Forest": rf_pipeline, "SVM": svm_pipeline, "XGBoost": xgb_pipeline, "ANN": ann_pipeline}
results = {"Logistic Regression": roc_log, "Random Forest": roc_rf, "SVM": roc_svm, "XGBoost": roc_xgb, "ANN": roc_ann}
best_model_name = max(results, key=results.get)
final_model = models[best_model_name]
print(f"Selected Model: {best_model_name}")


# Model Serialization
joblib.dump(model, "model.pkl")